In [ ]:
import os
import sys

import itertools
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats
import functools
import tabulate
import time
import importlib


sys.path.append("../")
from helper_DistNet import dist_helper, load_data_continuous

%matplotlib inline
print(os.getcwd())
plt.style.use('seaborn-v0_8-deep')
dpi=300

In [ ]:
ALGOS = ["DE", "DiagonalCMA", "modcma", "modde", "NaiveIsoEMNA", "NGOpt14", "NGOpt38", "OnePlusOne", "PSO", "RandomSearch", "RCobyla"]
DIMS = [5, 20]
BUDGETS = [100, 250, 500, 1000]  #, 2500, 5000, 10000]
RUNS = 50
DISTS = ["invgauss_floc", "norm", "expon_floc", "weibull_min_floc", "lognorm_floc"]

In [ ]:
#%%time
p_dict = dict()
lkh_dict = dict()
param_dict = dict()
SCENARIOS = list()

print(os.getcwd())
for algo in ALGOS:
    for budget in BUDGETS:
        for dim in DIMS:
            SC = f"{algo}_{dim}D_{budget}"
            p_dict[SC] = dict()
            lkh_dict[SC] = dict()
            param_dict[SC] = dict()
            print("Load %s %s" % (algo, budget))
            # Load performance data
            target_precs, _, instance_ls = load_data_continuous.get_data(
                algo=algo, 
                budget=budget, 
                dim=dim, 
                runs=RUNS, 
                agregate_runs=False, 
                log_target=False)
            if len(target_precs) == 0:
                print("No data found for %s %s" % (algo, budget))
                continue
            SCENARIOS.append(SC)
            print(target_precs.shape)
            for i in range(target_precs.shape[0]):
                y_max_ = np.max(target_precs[i, :])
                y_min_ = 0

                target_precs[i, :] = (target_precs[i, :] - y_min_) / y_max_
            
            for dist_name in DISTS:
                start = time.time()
                print("{:>20s}".format(dist_name), end="")
                p_dict[SC][dist_name] = list()
                lkh_dict[SC][dist_name] = list()
                param_dict[SC][dist_name] = list()

                for idx, instance in enumerate(target_precs):
                    assert len(instance) == 50
                    if idx%int(len(target_precs)/10) == 0: print(".",end="")
                    instance = np.array(instance)
                    param = dist_helper.fit_dist(x=instance, dist_name=dist_name)
                    param_dict[SC][dist_name].append(param)
                    
                    p = dist_helper.kstest(x=instance, dist_name=dist_name, param=param)
                    lkh = dist_helper.nllh(instance, param, dist_name)

                    p_dict[SC][dist_name].append(p)
                    lkh_dict[SC][dist_name].append(lkh)
                dur = time.time() - start
                print("%3.2gsec len: %3.2g mean_p: %3.2g, mean lkh: %3.2g" % (dur, len(target_precs), np.mean(p_dict[SC][dist_name]), np.mean(lkh_dict[SC][dist_name])))

In [ ]:
dist_trans = {
"expon_floc": "EXP",
"invgauss_floc": "INV",
"lognorm_floc": "LOG",
"norm": "N",
"weibull_min_floc": "WEI"
}
tabular_data = list()
TAB_SCENARIOS = list()
dim = 5
budget = 1000
for algo in ALGOS:
    TAB_SCENARIOS.append(f"{algo}_{dim}D_{budget}")

# for SC in TAB_SCENARIOS:
for SC in TAB_SCENARIOS:
    keys = sorted(DISTS)

    perc_ls = list()
    mean_ls = list()
    l_ls = list()
    d_ls = list()
    for dist_name in keys:
        #if "floc" in dist_name:
        #    continue
        num_instances = len(p_dict[SC][dist_name])
        num_equal = np.mean([1 if p <= 0.01 else 0 for p in p_dict[SC][dist_name]])*100
        perc_ls.append(num_equal)
        l_ls.append(np.mean(lkh_dict[SC][dist_name]))
        d_ls.append(dist_trans[dist_name])

    sort_idx = range(len(l_ls)) 
    # sort_idx = np.argsort(l_ls)
    
    TOP = len(DISTS)
    SC_name = SC.split("_")[0] # Only the algorithm name 
    SC_name = SC_name.replace("modde", "L-SHADE")
    SC_name = SC_name.replace("modcma", "Mod CMA")
    SC_name = SC_name.replace("DiagonalCMA", "Diagonal CMA")
    tabular_data.append([SC_name, ])
    # tabular_data.append([SC, ])
    # tabular_data[-1].extend([d_ls[i] for i in sort_idx[:TOP]])
    # tabular_data[-1].extend([d_ls[i] for i in sort_idx[:TOP]])
    # tabular_data.append([" ", ])
    tabular_data[-1].extend([round(l_ls[i], 3) for i in sort_idx[:TOP]])
    tabular_data[-1].extend([round(perc_ls[i],1) for i in sort_idx[:TOP]])

a = tabulate.tabulate(tabular_data, tablefmt="latex", floatfmt="7.4")
print(a)
os.makedirs("./results", exist_ok=True)
with open(f"./results/{dim}D_{budget}.tex", "w") as fh:
    fh.write(a)

In [ ]:
model_name = {
    "lognorm_floc": "LOGNORMAL",
    "invgauss_floc": "INVERSE GAUSSIAN",
    "weibull_min_floc": "WEIBULL",
    "expon_floc": "EXPONENTIAL",
    "norm": "NORMAL",
}
CUT_OFF = 5000
# for algo in ALGOS:
for algo in ["DE", "NGOpt14"]:
    for budget in [1000]:
        for dim in [5]:
            SC = f"{algo}_{dim}D_{budget}"
            keys = sorted(DISTS)

            perc_ls = list()
            mean_ls = list()
            l_ls = list()
            for dist_name in keys:
                #if "floc" in dist_name: continue
                num_instances = len(p_dict[SC][dist_name])
                num_equal = np.sum([1 if p > 0.05 else 0 for p in p_dict[SC][dist_name]])
                perc_ls.append(float(num_equal)/num_instances*100)
                mean_ls.append(np.mean(p_dict[SC][dist_name]))
                l_ls.append(np.mean(lkh_dict[SC][dist_name]))

            sort_idx = np.argsort(perc_ls)
            os.makedirs("./plots", exist_ok=True)
            print("{:>20}: {:>10s} {:>10s} {:>5s}".format("name", "% p>0.05", "mean p", "nll"))
            with open("./plots/distributions.txt", "a") as fh:
                fh.write("%s\n" % SC)
                fh.write("{:>20}: {:>10s} {:>10s} {:>5s}\n".format("name", "% p>0.05", "mean p", "nll"))
                for idx in reversed(sort_idx):
                    print("{:>20}: {: 10.2f} {: 10.3f} {: 1.3f}".format(keys[idx], perc_ls[idx], 
                                                                        mean_ls[idx], l_ls[idx]))
                    fh.write("{:>20}: {: 10.2f} {: 10.3f} {: 1.3f}\n".format(keys[idx], perc_ls[idx], mean_ls[idx], l_ls[idx]))
                fh.write("\n")
                # Plot fitted distribution
                target_precs, _, _ = load_data_continuous.get_data(
                        algo=algo, 
                        budget=budget, 
                        dim=dim, 
                        runs=RUNS, 
                        agregate_runs=False, 
                        log_target=False)
                
                for i in range(target_precs.shape[0]):
                    y_max_ = np.max(target_precs[i, :])
                    y_min_ = 0

                    target_precs[i, :] = (target_precs[i, :] - y_min_) / y_max_   
                
                x_axis = 10**np.arange(np.log10(0.000001), np.log10(CUT_OFF), 0.01)

                cs = ['#e41a1c','#377eb8','#ff7f00','#4daf4a','#984ea3','#a65628','#f781bf', "#37CA16", "#FF1493", '#999999',]
                colors1 = itertools.cycle(cs)
                colors2 = itertools.cycle(cs)

                num_plots = 2
                num_runs = 3
                for d_idx, s_idx in enumerate(reversed(sort_idx[-num_plots:])):
                    dist_name = keys[s_idx]
                    #if "floc" in dist_name: continue

                    plt.figure(figsize=(10, 4.5))
                    plt.rcParams.update({
                        "font.size": 16,      # default text size
                        "axes.titlesize": 25, # subplot titles
                        "axes.labelsize": 20, # x/y labels
                        "xtick.labelsize": 18,
                        "ytick.labelsize": 18,
                        "legend.fontsize": 18
                    })
                    min_ = 100
                    max_ = -100
                    for idx, instance in enumerate(target_precs[:num_runs]):
                        srtd = np.sort(instance)
                        yvals = np.arange(1,len(srtd)+1)/float(len(srtd))
                        min_= min(np.min(instance), min_)
                        max_= max(np.max(instance), max_)
                        plt.step(srtd, yvals, c=next(colors1), label='empirical CDFs' if idx == 0 else None)
                        #plt.hist(instance, normed=True)

                    for idx, param in enumerate(param_dict[SC][dist_name][:num_runs]):
                        y = dist_helper.lhood_dist(x_axis, dist_name=dist_name, param=param)
                        y = dist_helper.cdf_dist(x_axis, dist_name=dist_name, param=param)
                        plt.plot(x_axis, y, c=next(colors2), linestyle='--', label=f'fitted distribution' if idx == 0 else None)
                    
                    SC_name = SC.replace("modde", "L-SHADE")
                    SC_name = SC_name.replace("modcma", "Mod CMA")
                    SC_name = SC_name.replace("DiagonalCMA", "Diagonal CMA")
                    plt.xscale("log")
                    plt.xlim([min_, max_])
                     # set fixed number of x ticks (e.g., 6)
                    ax = plt.gca()
                    ax.xaxis.set_minor_locator(plt.NullLocator())
                    ticks = np.logspace(np.log10(min_), np.log10(max_), 5)
                    plt.xticks(ticks, labels=[f"{t:.2f}" for t in ticks])

                    # labels = [f"${ticks[0]*10**8:.1f}\\times 10^8$", 
                    #           f"${ticks[1]*10**7:.1f}\\times 10^7$", 
                    #           f"${ticks[2]*10**5:.1f}\\times 10^5$", 
                    #           f"${ticks[3]*10**3:.1f}\\times 10^3$", 
                    #           f"${ticks[4]:.1f}$"]
                    # plt.xticks(ticks, labels=labels)
                    #plt.ylim([-0.01, 1.01])
                    plt.ylabel("CDF")
                    plt.xlabel("Log10 regret (normalized)")
                    plt.legend()
                    plt.tight_layout()
                    os.makedirs(f"plots/thesis/additional", exist_ok=True)
                    plt.savefig(f"plots/thesis/additional/fitted_distributions_{SC}_{model_name.get(dist_name, dist_name)}.pdf" )
                    plt.close()

In [ ]:
#dist_trans = {
#"expon_floc": "EXP",
#"invgauss_floc": "INV",
#"lognorm_floc": "LOG",
#"norm": "N",
#"weibull_min_floc": "WEI"
#}
dist_name = "norm"

plt.figure(figsize=(5, len(SCENARIOS)*2))
plt_idx = 0
cs = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#a65628','#f781bf','#999999']
colors1 = itertools.cycle(cs)

for SC in SCENARIOS:
    p = np.array(param_dict[SC][dist_name])
    
    for param_idx in (0,1):
        plt_idx += 1
        plt.subplot(len(SCENARIOS), 2, plt_idx)
        if plt_idx == 1: plt.title("mu")
        if plt_idx == 2: plt.title("scale")
        if plt_idx%2 == 1: plt.ylabel(SC)
        plt.hist(p[:, param_idx], facecolor=next(colors1), density=True)
            
plt.tight_layout()
plt.savefig("plots/%s_parameter_distribution.png" % dist_name)
plt.show()

## Generate plots of goodness-of-fit test

In [ ]:
import pandas as pd
import seaborn as sns


dist_trans = {
"expon_floc": "EXP",
"invgauss_floc": "INV",
"lognorm_floc": "LOG",
"norm": "N",
"weibull_min_floc": "WEI"
}
results = []
for dist_name in DISTS:
    mean_ks = np.mean([np.mean([1 if p <= 0.01 else 0 for p in p_dict[sc][dist_name]])*100 for sc in SCENARIOS])
    mean_nllh = np.mean([np.mean(lkh_dict[sc][dist_name]) for sc in SCENARIOS])
    results.append({
        "Distribution": dist_trans.get(dist_name, dist_name), 
        "Avg_KS": mean_ks, 
        "Avg_NLLH": mean_nllh})

df = pd.DataFrame(results)
plt.rcParams.update({
    "font.size": 16,      # default text size
    "axes.titlesize": 20, # subplot titles
    "axes.labelsize": 20, # x/y labels
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 16
})
# --- Plot NLLH ranking ---
plt.figure(figsize=(9,5))
sns.barplot(data=df.sort_values("Avg_NLLH"), x="Distribution", y="Avg_NLLH", palette="viridis", hue="Distribution")
plt.title("Average NLLH across Scenarios")
plt.ylabel("Average NLLH")
plt.tight_layout()
plt.savefig("plots/Avg_NLLH_ranking.pdf", dpi=dpi)

# --- Plot KS ranking ---
plt.figure(figsize=(9,5))
sns.barplot(data=df.sort_values("Avg_KS"), x="Distribution", y="Avg_KS", palette="mako", hue="Distribution")
plt.title("Average KS Rejection Rate (%) across Scenarios")
plt.ylabel("Average KS (%)")
plt.tight_layout()
plt.savefig("plots/Avg_KS_ranking.pdf", dpi=dpi)